In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

In [ ]:
# Load dataframe
df = pd.read_csv("../data/raw/listings.csv")

Load relevant columns of the more detailed dataframe, based on domain-knowledge.

The information on the number of beds is not selected, because it is unreliable on AirBNB, as the definition of a bed is ambiguous. `accommodates` captures the sleeping capacity more reliably.

In [ ]:
# Load relevant columns of the more detailed dataframe, based on domain-knowledge
detail_ratings = ["review_scores_cleanliness", "review_scores_checkin", 
               "review_scores_communication", "review_scores_location", 
               "review_scores_accuracy", "review_scores_value"]

add_df = pd.read_csv(
    "../data/raw/listings.csv.gz",
    usecols=["id"] + detail_ratings + ["review_scores_rating", "host_is_superhost", "bedrooms", "accommodates", "instant_bookable"]
)

In [ ]:
# Merge both dataframes
df = df.merge(
    add_df,
    on="id",
    how="left"
)
df.head(3)

## Data Cleaning

### Column Selection

`neighbourhood_group` has no existing values, therefore remove it.

In [ ]:
not_null_count = df["neighbourhood_group"].notna().sum()
print(f"Number of existing values: {not_null_count}")

# Remove column
df = df.drop(columns=["neighbourhood_group"])

Remove columns with no useful information on the price (except the id):

In [ ]:
df = df.drop(columns=["name", "host_id", "host_name", "license"])

Remove columns based on domain knowledge:

In [ ]:
df = df.drop(columns = ["minimum_nights", "last_review", "calculated_host_listings_count", "availability_365"])

`reviews_per_month` is redundant because of `number_of_reviews`, therefore remove it.

In [ ]:
df = df.drop(columns="reviews_per_month")

When the overall rating `review_scores_rating` is available, then all the more detailed ratings are also available:

In [ ]:
mask = df["review_scores_rating"].notna()
print("Number of missing values of detailed ratings, where overall rating exists:")
print(df.loc[mask, detail_ratings].isna().sum())

The overall rating `review_scores_rating` includes the price-performance ratio `review_scores_value`, which is directly influenced by price and would introduce reverse causality.

Because of the completeness of the more detailed ratings, I choose to use the average of all detailed ratings, excluding `review_scores_value`, instead of `review_scores_rating`:

In [ ]:
# Compute custom rating
rating_cols = [col for col in detail_ratings if col != "review_scores_value"]
df["avg_rating"] = df[rating_cols].mean(axis=1)

# Remove other ratings
df = df.drop(columns=detail_ratings+["review_scores_rating"])

Looking at missing values of the rating, we can observe that all listings without review have a missing rating. They are imputed with 0 and the binary variable `has_reviews` is added to indicate the absence of reviews rather than a low rating.
 
Both variables will be included in the linear regression model.

In [ ]:
# Compute number of listings without review, that have a rating
num = df.loc[df["number_of_reviews"]==0, "avg_rating"].notna().sum()
print(f"{num} listings with no review have a rating.") 

# Creating indicator variable
df["has_reviews"] = (df["number_of_reviews"]>0)

# Imputing null-values with 0
df.loc[~df["has_reviews"], "avg_rating"] = 0

`number_of_reviews_ltm` is the number of reviews in the last twelve months and is more current than the total `number_of_reviews`, which is therefore removed.

In [ ]:
# Drop `number_of_reviews`
df = df.drop(columns=["number_of_reviews"])

`neighbourhood` captures the geographic information of listings and makes `longitude` and `latitude` redundant, as shown in the scatter plot below. 

I will rather use this variable than to compute the distance to the city center, because neighbourhood boundaries likely reflect price differences better than raw distance. Therefore, `longitude` and `latitude` are removed.

In [ ]:
# Plot neighbourhoods by geographic location
sns.scatterplot(
    x="longitude", 
    y="latitude", 
    hue="neighbourhood",
    data=df,
    s=10
)
plt.title("Neighbourhoods of Florence")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

# Remove geographic information of listings
df = df.drop(columns=["longitude", "latitude"])

Reorder columns

In [ ]:
# Reorder dataframe columns into a meaningful order
df = df[["id", "neighbourhood", "room_type", "accommodates", 
         "bedrooms", "host_is_superhost", "instant_bookable",
         "number_of_reviews_ltm", "has_reviews", "avg_rating",
         "price"]]
df.head(3)

### Data Type Conversion

In [ ]:
# Convert best possible Pandas dtypes
df = df.convert_dtypes()

# `host_is_superhost` and `instant_bookable` as boolean
bool_map = {"t": True, "f": False}
df["host_is_superhost"] = df["host_is_superhost"].map(bool_map).astype("boolean")
df["instant_bookable"] = df["instant_bookable"].map(bool_map).astype("boolean")

# String to category
df["neighbourhood"] = df["neighbourhood"].astype("category")
df["room_type"] = df["room_type"].astype("category")

df.head(3)

### Target Variable (Price)

#### Missing Values

Around 9% of price values are missing. A comparison of the feature-distributions showed only small differences and no strong indication of systematic bias.

Because imputing a target variable is not sensible, the rows with missing price values are dropped.

In [ ]:
missing_percent = df["price"].isna().mean()*100
print(f"Missing price values: {missing_percent:.1f}%")

# Remove rows with missing price values
df = df.dropna(subset=["price"])

#### Outliers

In [ ]:
quantiles = np.arange(0.93, 1.0, 0.005)
values = df["price"].quantile(quantiles)
plt.plot(quantiles, values)
plt.xlabel("Quantile")
plt.ylabel("Price")
plt.title("Price distribution at high quantiles")
plt.show()

Listings above the 98.5th percentile are removed, because the quantile plot shows a steep increase after this point, indicating luxury properties that are not representative of typical Airbnb listings in Florence.

In [ ]:
q985 = df["price"].quantile(0.985)
df = df[df["price"]<=q985]

For economic target variables like `price`, the underlying relations are often multiplicative, which leads to a right-skewed distribution in the data. Applying the logarithm transforms these into additive relations. The histogram below confirms the target-variable `price` is right-skewed. 

The log-transformation also reduces the impact of outliers, which is particularly important for the target variable because linear regression minimizes the squared residuals of the target variable.

It also makes the distribution more symmetric, which makes it more likely to meet the regression assumptions of normally distributed residuals. These assumptions will still be verified after fitting the model.

For these reasons, I will use `log_price` as the target variable during the exploration and analysis.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
df["log_price"] = np.log(df["price"])

sns.histplot(df["price"], bins=45, kde=True, ax=ax[0])
ax[0].set_xlabel("Price")
ax[0].set_ylabel("Count")
ax[0].set_title("Target variable (Price) is right-skewed")

sns.histplot(df["log_price"], bins=45, kde=True, ax=ax[1])
ax[1].set_xlabel("Log(Price)")
ax[1].set_ylabel("Count")
ax[1].set_title("Log(Price) has a more symmetric distribution")

plt.tight_layout()
plt.show()

### Handling Outliers

All values of numerical variables appear realistic.

Influential outliers will be identified after fitting the initial linear regression model using Cook's Distance and removed if neccessary.

In [ ]:
df[["accommodates", "bedrooms", "number_of_reviews_ltm", "avg_rating"]].describe()

### Handling Missing Values

#### Accommodates

#### Bedrooms

#### Host is superhost

#### Instant bookable

#### Avg rating

#### Neighbourhood

## Save Preprocessed Data to File

In [ ]:
df.to_csv("../data/processed/florence_processed.csv", index=False)